### Graph Synonym:

In [1]:
from dotenv import load_dotenv
import os
import pandas as pd

#from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from source.Neo4jPropertyGraphStore import Neo4jPropertyGraphStore
#from llama_index.core import PropertyGraphIndex
from source.PropertyGraphIndex import PropertyGraphIndex

from source.LLMSynonymRetriever import LLMSynonymRetriever
from source.VectorContextRetriever import VectorContextRetriever

from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
#from llama_index.core.indices.property_graph import SchemaLLMPathExtractor, SimpleLLMPathExtractor
from llama_index.core.schema import TextNode

from source.SimpleLLMPathExtrator import SimpleLLMPathExtractor

import nest_asyncio

nest_asyncio.apply()
load_dotenv(dotenv_path='/home/duy/workspace/RAG_Traffic_Law/.env')

True

In [2]:
DATA_SAMPLE = '/home/duy/workspace/RAG_Traffic_Law/sample_data/sample.csv'
llm = OpenAI(model="gpt-4o", api_key=os.getenv("OPENAI_API_KEY")) 
embed_model = OpenAIEmbedding(model="text-embedding-3-large", api_key=os.getenv("OPENAI_API_KEY"))

### Load graph store:

In [3]:
graph_store = Neo4jPropertyGraphStore(
    username="neo4j",
    password="Tgs4ZghUP8hFMKQ",
    url="bolt://localhost:7687",
    database="neo4j",
)

index = PropertyGraphIndex.from_existing(
    property_graph_store=graph_store,
    llm=llm,
    embed_model=embed_model,
)

### Load retriever:

In [8]:
Synonym_Retriever = LLMSynonymRetriever(
    graph_store=graph_store,
    include_text=False,
    llm=llm,
    embed_model=embed_model,
    path_depth = 2  
)

In [9]:
retriever = index.as_retriever(
    sub_retrievers = [Synonym_Retriever],
    include_text=False,  # include source text in returned nodes, default True
)

In [10]:
nodes = retriever.retrieve("traffic sign")
print(nodes)
for node in nodes:
    print(node)

ic| response: 'traffic sign^traffic signs^road sign^road signs^street sign^street signs^highway sign^highway signs^traffic signal^traffic control device'
ic| matches: ['traffic sign',
              'traffic signs',
              'road sign',
              'road signs',
              'street sign',
              'street signs',
              'highway sign',
              'highway signs',
              'traffic signal',
              'traffic control device']
ic| 'Get Match'
ic| cypher_statement: '''MATCH (e)WHERE e.name IS NOT NULL AND e.name in $names 
                               WITH e
                               RETURN e.name AS name,
                                      [l in labels(e) WHERE l <> '__Entity__' | l][0] AS type,
                                      e{.* , embedding: Null, id: Null} AS properties
                               '''
ic| params: {'names': ['traffic sign',
                       'traffic signs',
                       'road sign',
                  

[NodeWithScore(node=TextNode(id_='297e7149-10c0-4925-b740-89c7eb031cb5', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='400.000->600.000 VND -> VIOLATION_BEHAVIOUR -> traffic sign', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=1.0), NodeWithScore(node=TextNode(id_='eb0ac098-6027-4dfa-881f-2af34d5094f6', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Motor vehicle drivers -> REGULATE -> 400.000->600.000 VND', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=1.0), NodeWithScore(node=TextNode(id_='e7764d71-881f-453b-886b-6805cb434215', embedding=Non

In [11]:
nodes

[NodeWithScore(node=TextNode(id_='297e7149-10c0-4925-b740-89c7eb031cb5', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='400.000->600.000 VND -> VIOLATION_BEHAVIOUR -> traffic sign', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=1.0),
 NodeWithScore(node=TextNode(id_='eb0ac098-6027-4dfa-881f-2af34d5094f6', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Motor vehicle drivers -> REGULATE -> 400.000->600.000 VND', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=1.0),
 NodeWithScore(node=TextNode(id_='e7764d71-881f-453b-886b-6805cb434215', embedding=N